98. Validate Binary Search Tree
Medium

Given the root of a binary tree, determine if it is a valid binary search tree (BST).

A valid BST is defined as follows:

    The left subtree of a node contains only nodes with keys strictly less than the node's key.
    The right subtree of a node contains only nodes with keys strictly greater than the node's key.
    Both the left and right subtrees must also be binary search trees.


#### Analysis 

1. The key is that every node must satisfy constraints from all its ancestors—not just its immediate parent.

root = [5, 1, 7, None, None, 4, 8]

Here, 7 is the right child of 5, and 4 is the left child of 7.

Although 4 < 7, 4 is also in the right subtree of 5, so it must be greater than 5. This tree is invalid.

We need to verify:
- Every value in a node’s left subtree is strictly smaller.
- Every value in its right subtree is strictly greater.
- These rules hold throughout the tree.
- Duplicates are invalid because the comparisons are strict.
Checking only these conditions is insufficient:
    node.left.val < node.val
    node.right.val > node.val

2. Naive approach: repeatedly inspect entire subtrees

For every node:
1. Traverse its entire left subtree and check that all values are smaller.
2. Traverse its entire right subtree and check that all values are larger.
3. Repeat those checks at each child.
This works, but repeatedly visits the same descendants.

Time: O(n²) worst case.
Auxiliary space: O(h) using DFS, where h is the tree’s height.

We can eliminate repeated scans by carrying the ancestor constraints as we traverse.

3. Optimal approach: DFS with lower and upper bounds

Each node has an allowed interval:
    lower < node.val < upper

Start the root with no restrictions:
    lower = float("-inf")
    upper = float("inf")

When moving to a child:
| Move | New allowed range | Reason |
|---|---|---|
| Left child | `(lower, node.val)` | Must be smaller than the current node |
| Right child | `(node.val, upper)` | Must be greater than the current node |

Keep the other bound unchanged because it represents an ancestor’s constraint.

Use a stack of (node, lower, upper) tuples for iterative DFS. No visited set is needed for a proper binary tree.


In [1]:
# Optimal solution

class Solution:
    def isValidBST(self, root) -> bool:
        if root is None:
            return True

        stack = [(root, float("-inf"), float("inf"))]

        while stack:
            node, lower, upper = stack.pop()

            # Strict bounds also reject duplicate values.
            if not lower < node.val < upper:
                return False

            # Right subtree inherits the upper bound.
            if node.right:
                stack.append((node.right, node.val, upper))

            # Left subtree inherits the lower bound.
            if node.left:
                stack.append((node.left, lower, node.val))

        return True
## usage 
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Root 2, left child 1, right child 3.
root = TreeNode(2, TreeNode(1), TreeNode(3))

solution = Solution()  # The optimal Solution class from above.
print(solution.isValidBST(root))  # True

root = TreeNode(
    5,
    TreeNode(1),
    TreeNode(
        7,
        TreeNode(4),
        TreeNode(8)
    )
)

print(solution.isValidBST(root))  # False

True
False


Another optimal approach: inorder traversal

A BST’s inorder traversal—left, node, right—must produce a strictly increasing sequence.

For the invalid example, inorder produces:

In [ ]:
class Solution:
    def isValidBST(self, root) -> bool:
        stack = []
        current = root
        previous = None

        while current or stack:
            # Reach the leftmost unprocessed node.
            while current:
                stack.append(current)
                current = current.left

            current = stack.pop()

            # Inorder values must increase strictly.
            if previous is not None and current.val <= previous:
                return False

            previous = current.val
            current = current.right

        return True

Both approaches have:
- Time: O(n) — visit each node at most once.
- Auxiliary space: O(h) — O(log n) for a balanced tree, O(n) worst case.
For explaining correctness in an interview, the bounds approach directly expresses the problem’s rules. The inorder version is useful if you already remember iterative inorder traversal.

Essential edge cases

| Tree in level order | Expected | Reason |
|---|---|---|
| `[2]` | `True` | Single node |
| `[2, 1, 3]` | `True` | Valid ordering |
| `[2, 2, 3]` | `False` | Duplicate value |
| `[5, 1, 7, None, None, 4, 8]` | `False` | Descendant violates an ancestor’s bound |
| `[0, -1, 1]` | `True` | Negative values and zero are valid |
| `[1, None, 2, None, 3]` | `True` | A skewed tree can still be a BST |

In [ ]:
# Naive solution 

def is_valid_bst_naive(root):

    def all_less(node, limit):
        # An empty subtree has no violating values.
        if node is None:
            return True

        return (
            node.val < limit
            and all_less(node.left, limit)
            and all_less(node.right, limit)
        )

    def all_greater(node, limit):
        if node is None:
            return True

        return (
            node.val > limit
            and all_greater(node.left, limit)
            and all_greater(node.right, limit)
        )

    def validate(node):
        if node is None:
            return True

        return (
            all_less(node.left, node.val)
            and all_greater(node.right, node.val)
            and validate(node.left)
            and validate(node.right)
        )

    return validate(root)


The same nodes are scanned repeatedly.
- Time: O(n²) worst case.
- Auxiliary space: O(h) for recursion.
- A deeply skewed tree can hit Python’s recursion-depth limit.
The optimal bounds solution checks each node just once.